In [1]:
# Notebook only: let Python find `agent.py` and the `utils/` and `tools/` folders.
# This notebook lives in 01_kyc_agent/modules/, so the 01_kyc_agent folder is one level up.
import sys
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool, ToolRuntime
from models import model

# if __name__ == "__main__":
#     print(init_chat_model(model).invoke("Say hello to the KYC hackathon in one sentence.").content)

Hello, KYC hackathon—let’s build something great!


In [4]:
import importlib
import utils.state

importlib.reload(utils.state)

<module 'utils.state' from '/workspaces/finsights/agents/utils/state.py'>

In [3]:
from langchain.agents import create_agent
from utils.state import InvestigationContext

In [4]:
from utils.state import get_company, get_date

In [5]:
system_prompt=(project_root/ "supervisor_agent" / "prompt.md").read_text()
system_prompt

"You are a risk management assitant.\n\nEvery day you analyse the company you are invesitgating's price movements and then analyse the results using the tools provided.\n\nReview the data and produce a brief summary indicating what might be behind the price movements and if this "

In [14]:
from price_intelligence_agent.agent import agent as price_agent

In [16]:
@tool
def call_price_agent(
    query: str,
    runtime: ToolRuntime[InvestigationContext]
) -> str:
    """Delegate financial analysis tasks to the financial analyst."""

    result = price_agent.invoke(
        {
            "messages": [
                {"role": "user", "content": query}
            ]
        },
        context=runtime.context
    )

    return result["messages"][-1].content

In [18]:
supervisor_agent = create_agent(
    model=model,
    system_prompt=system_prompt,
    context_schema=InvestigationContext,
    tools=[get_company, get_date, call_price_agent]
)

In [23]:
response = supervisor_agent.invoke(
{"messages": [{"role": "user", "content": "What can you tell me about price movements?"}]},
context=InvestigationContext(company_name='NVDA', date_str='2026-11-19')
)

print(response["messages"][-1].content)

[{'id': 'rs_0927be0339e476f6006b06d8cc949c87d1946ab39dc6a531f8', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABrBtjOXiyjitTXvHFhK1WdmWdJTQsXqElTSQmpgCyimMZYKIG3kEgchH6gDMo19O-t-DV-4gICG8YlvfUPwQAwzKGVG8ubqUZYk-a2nHDQsv9FukXrBTx1bsz1pH-YIdMm2_q7fka1lZlerOUTSqHKFBshl13F_V2KEyFezhj-hYkPINeID_xj3W01LZbLB1mrTOMBlMRa1HPee5y-ArrUf0spZgFxMqUW7DsqArDof_eV_qRPB6ykr3RROirhL1MVqUkRoOfi8l1N-Wshz7O9QGWR4oD4AN-U7giDJQt76j_PTXFEtf3oQjpWzqW8mvoYzYjWjiqRy6cAtu6coeNLUO6PVVQEt75TwM5R3fNegU44DNbnBMCxDFlwsvOobrJUEQcVeFmMt1ZAuXwdBsuMxKv-gqMBG9RpePS7HOFVwdO201yxIGpjdSEOgDafatyk6K0RGVBBj0AIEiykgvfa1QqPwyG-H46aIciZmOqy9CdKGfZVKd5F0hoUvH1RTu130DHM6C4BzvvRA4FVn8JgoQaxChY_bNjMN9d6F4PNLiIgBVswTBXKSOr55boLFg7q5BoWSBYrJmBch1xWNx-5A8ARzhXcVTp0KAxpp_BC0A3ltM9SXqlc5M_9fFpqcpkDP7p8w9xrhAQHDi48R6xLexjjzziFsj9oYDxISz_3YZYl_KPeXl8f1tRNN6Pzi8yshvrxgaXYU_QwqUZ0t_aC9IlaJGVj6fGkmtzG9UfL9KXTUyh_laadbfwP3pxDxciRXinlTh1tJvn6YJ2ZRzp0HhbPKv8LqB6fFxjVykOs4LdjJ8X1EPcMeOpu-ytlIgFkdedJLKYd6pcVTb-XcuaNg8

In [22]:
response['messages']

[HumanMessage(content='What can you tell me about price movements?', additional_kwargs={}, response_metadata={}, id='a96a5813-3b90-4050-b05f-61f01e13d228'),
 AIMessage(content=[{'arguments': '{}', 'call_id': 'call_248a75d9003a4680b68cfb8c77c2a963', 'name': 'get_company', 'type': 'function_call', 'id': 'fc_09532480740d387b006b06d82e3fa487d1850fa2ed3ed28fda', 'status': 'completed'}, {'arguments': '{}', 'call_id': 'call_43d1333c690342a290a8cd4a3d992873', 'name': 'get_date', 'type': 'function_call', 'id': 'fc_09532480740d387b006b06d82e5fa887d185d15a574f7a4929', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_09532480740d387b006b06d82d7a5887d198aea00152428553', 'created_at': 1795610669.0, 'metadata': {}, 'model': 'gpt-6-luna', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-6-luna'}, id='resp_09532480740d387b006b06d82d7a5887d198aea00152428553', tool_calls=[{'name': 'get_company', 'args': {},